In [ ]:
import os
import json
import pandas as pd
import numpy as np

In [ ]:
files = {
    "warehouse1": [
        "warehouse-20-40-10-2-1-random-1-k50_paths_2026-03-16_seed123_k50",
    ],
    "maze1": [
        "maze-128-128-1-even-1-k20_paths_2026-03-16_seed123_k20"
    ]
}
all_results = {}
for map_name in files:
    all_results[map_name] = {}
    for filename in files[map_name]:
        num_agents = filename.split("_")[-1]
        filepath = os.path.join(os.path.dirname(os.path.abspath("__file__")), "output", map_name, f"{filename}.json")
        all_results[map_name][num_agents] = json.load(open(filepath))

In [ ]:
df = pd.DataFrame(columns=["map", "num_agents", "num_agents_tested", "avg_search_time_flexsipp", "avg_generation_time_flexsipp", "avg_found_paths_flexsipp", "avg_search_time_maeder", "avg_generation_time_maeder", "avg_found_paths_maeder", "avg_delay_improvement"])
num_rows = 0
for map_name in all_results:
    for num_agents in all_results[map_name]:
        maeder = {"search": [], "generation": [], "paths": [], "delays": {a: [] for a in all_results[map_name][num_agents]}}
        flexsipp = {"search": [], "generation": [], "paths": [], "delays": {a: [] for a in all_results[map_name][num_agents]}}
        for delay_agent in all_results[map_name][num_agents]:
            if "FlexSIPP" in all_results[map_name][num_agents][delay_agent]:
                flexsipp["search"].append(all_results[map_name][num_agents][delay_agent]["FlexSIPP"]["Search Time"])
                flexsipp["generation"].append(all_results[map_name][num_agents][delay_agent]["FlexSIPP"]["gen_time"])
                flexsipp["paths"].append(len(all_results[map_name][num_agents][delay_agent]["FlexSIPP"]["unique_routes_safe"]))
                for path, atf_strings in all_results[map_name][num_agents][delay_agent]["FlexSIPP"]["unique_routes_safe"].items():
                    for atf_str in atf_strings:
                        atf = [float(x) if "inf" not in x else np.inf for x in atf_str.replace("(", "").replace(")", "").split(", ")]
                        current_delay = (atf[1] + atf[3]) - all_results[map_name][num_agents][delay_agent]["original_arrival_time"]
                        for t, t_atf, t_path, other_delays in all_results[map_name][num_agents][delay_agent]["FlexSIPP"]["tipping_points"]:
                            if t_path == path:
                                current_delay += sum([min([d for x, d in other_delays[a].items()]) if other_delays[a] else 0 for a in other_delays])
                        flexsipp["delays"][delay_agent].append(current_delay)
                maeder["search"].append(all_results[map_name][num_agents][delay_agent]["@MAEDeR"]["Search Time"])
                maeder["generation"].append(all_results[map_name][num_agents][delay_agent]["@MAEDeR"]["gen_time"])
                maeder["paths"].append(len(all_results[map_name][num_agents][delay_agent]["@MAEDeR"]["unique_routes_safe"]))
                for path, atf_strings in all_results[map_name][num_agents][delay_agent]["@MAEDeR"]["unique_routes_safe"].items():
                    for atf_str in atf_strings:
                        atf = [float(x) if "inf" not in x else np.inf for x in atf_str.replace("(", "").replace(")", "").split(", ")]
                        current_delay = (atf[1] + atf[3]) - all_results[map_name][num_agents][delay_agent]["original_arrival_time"]
                        maeder["delays"][delay_agent].append(current_delay)
        df.loc[num_rows] = [
            map_name, 
            num_agents, 
            len(maeder["search"]), 
            sum(flexsipp["search"]) / len(flexsipp["search"]), 
            sum(flexsipp["generation"]) / len(flexsipp["generation"]), 
            sum(flexsipp["paths"]) / len(flexsipp["paths"]), 
            sum(maeder["search"]) / len(maeder["search"]), 
            sum(maeder["generation"]) / len(maeder["generation"]), 
            sum(maeder["paths"]) / len(maeder["paths"]),
            sum([min(maeder["delays"][a] if maeder["delays"][a] else [0]) - min(flexsipp["delays"][a]  if flexsipp["delays"][a] else [0]) for a in flexsipp["delays"]]) / len(flexsipp["delays"])
        ]
        num_rows += 1
df